In [78]:
from turtle import pu
import requests
import time
import pandas as pd
import random
import html

# Download dicty literatures from EPMC

this contains interactive notebook from Jaka, in the end in the production I used article_fetch/ from Jakob

## query EPMC 

In [2]:
BASE_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

In [23]:
def search(query, cursor_mark):
    params = {
        'resultType': 'core',
        'format': 'json',
        'synonym': 'N',           # exact keyword
        'query': query,
        'pageSize': 1000,  
        'cursorMark': cursor_mark,
    }

    return requests.get(BASE_URL, params=params)

In [24]:
url_params = '''
(
   dictyostelium   
) 
AND (
    (HAS_FT:Y) AND 
    (SRC:MED OR SRC:PMC )
)
'''
pubmed_ids = []
cursor_mark = '*'

In [19]:
# more filters
# url_params = '''
# (
#    dictyostelium   
# ) 
# AND (
#     (FIRST_PDATE:[2000 TO 2025]) AND
#     (HAS_FT:Y) AND 
#     (((SRC:MED OR SRC:PMC OR SRC:AGR OR SRC:CBA) NOT (PUB_TYPE:"Review")))
# )
# '''

In [25]:
while True:
    response = search(url_params, cursor_mark)

    if len(pubmed_ids) == 1000: # <------------------------------remove this limit to get all results
        break 

    if response.status_code != 200:
        print(response.text)
        break
    data = response.json()
    print(data['hitCount'], len(data['resultList']['result']))

    if len(data['resultList']['result']) == 0:
        break

    for result in data['resultList']['result']: 
        if result['source'] in ('MED', 'PMC'):

            journal_info = result.get('journalInfo')
            journal = None
            if journal_info:
                # Medline abbreviation if available, otherwise journal title
                if 'medlineAbbreviation' in journal_info['journal']:
                    journal = journal_info['journal']['medlineAbbreviation']
                else:
                    journal = journal_info['journal']['title']


            pub_type = result['pubTypeList']['pubType']
            # if 'Abstract' in pub_type:
            #     continue

            # is_review = 'review-article' in pub_type or 'Review' in pub_type

            is_open_access = result['isOpenAccess'] == 'Y'
            cited_by_count = result['citedByCount']
            pub_year = result.get('pubYear')
            language = result.get('language')
            pub_type_list = ';'.join(result['pubTypeList']['pubType'])

            if 'keywordList' in result:
                keywords = ';'.join([keyword if keyword is not None else '' for keyword in result['keywordList']['keyword']])

            else:
                keywords = None

            # print(result['pmid'], result.get('pmcid'), pub_date, medline_abbreviation)
            pubmed_ids.append(
                (
                    # result.get('pmid'),
                    result.get('pmcid'),
                    int(is_open_access),
                    int(cited_by_count),
                    language,
                    pub_type_list,
                    pub_year,
                    journal,
                    result.get('title'),
                    keywords
                )
            )

    if 'nextCursorMark' not in data:
        break

    cursor_mark = data['nextCursorMark']

    time.sleep(random.uniform(0.5, 1.5))

# print(
#     f'Year: {year}, Number of articles saved: {len(pubmed_ids)} out of total {data["hitCount"]}'
# )

16027 1000


In [26]:
df = pd.DataFrame(
    pubmed_ids,
    columns=[
        # 'pmid',
        'pmcid',
        'is_open_access',
        'cited_by_count',
        'language',
        'pub_type_list',
        'pub_year',
        'journal',
        'title',
        'keywords',

    ],
)

In [27]:
df

,pmcid,is_open_access,cited_by_count,language,pub_type_list,pub_year,journal,title,keywords
0,PMC12572271,1,0,eng,research-article;Journal Article,2025,Sci Rep,Dictyostelium exhibits PCB-induced impairment ...,Iron homeostasis;Dictyostelium;Cellular Toxici...
1,PMC12143694,1,0,eng,research-article;Journal Article,2025,Cell Adh Migr,Nhe1 is required for directional sensing in ve...,Chemotaxis;Dictyostelium;cell migration;Nhe1;E...
2,PMC12505270,1,1,eng,research-article;Journal Article,2025,Biol Open,Differential PaxillinB dynamics at Dictyosteli...,Dictyostelium discoideum;cell migration;Adhesi...
3,PMC12547847,1,0,eng,Historical Article;other;Interview,2025,Biol Open,First person - Julio Fierro Morales.,None
4,PMC12362907,1,0,eng,Introductory Journal Article;Editorial,2025,BMC Mol Cell Biol,Cell biology of Dictyostelium.,None
...,...,...,...,...,...,...,...,...,...
995,PMC11180820,1,5,eng,review-article;Review;Journal Article,2024,Front Cell Dev Biol,Methods and computational tools to study eukar...,Chemotaxis;Trajectory analysis;cell migration;...
996,PMC12415853,1,0,eng,research-article;Journal Article,2025,Synth Biol (Oxf),GoldenBraid2.0 &lt;i&gt;E. coli&lt;/i&gt;: a c...,Genetic engineering;Cloning;Molecular;Gram-neg...
997,PMC7192593,1,4,eng,"Research Support, Non-U.S. Gov't;research-arti...",2020,Nucleic Acids Res,"DIRS retrotransposons amplify via linear, sing...",None
998,PMC6594445,1,16,eng,"Research Support, Non-U.S. Gov't;research-arti...",2019,Mol Biol Cell,Ate1-mediated posttranslational arginylation a...,None


In [28]:
summary = (df["pub_year"]
           .value_counts(dropna=False)
           .rename_axis("pub_year")
           .reset_index(name="count")
           .sort_values("pub_year"))

summary

,pub_year,count
8,2018,1
6,2019,37
5,2020,58
4,2021,111
3,2022,112
2,2023,142
1,2024,235
0,2025,303
7,2026,1


In [1]:
df.to_csv(f'yun-dicty.csv', index=False)

9767 1000


## The below code will be used to get the BioC format for the articles and store them on disk.

files will be saved as pmcid.json.gz

In [1]:
import requests
import time
import json 
import random
import os
import pandas as pd
import gzip


pmcids = list(df['pmcid'])[:5] # <------------------------------remove this limit to get all results


# # print(len(missing_pmcids - processed))

for pmcid in pmcids:
    url = f'https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode'
    response = requests.get(url)
    if response.status_code == 200:
        if 'No record can be found for the input:' in response.text or 'No result can be found' in response.text:
            print(f'No record for {pmcid}', flush=True)
            continue
        with gzip.open(f'{pmcid}.json.gz', 'wt') as f:
            f.write(json.dumps(response.json()[0], indent=2))
        print(f'{pmcid} saved', flush=True)
        # print(response.json())
    else:
        print(f'Error: {response.status_code}', flush=True)
    
    time.sleep(random.uniform(2, 5))

NameError: name 'df' is not defined

# literatures on dictybase

## enl file
load enl file from dictybase web page

In [8]:
from pathlib import Path
import zipfile
import re
import pandas as pd
import xml.etree.ElementTree as ET
from collections import Counter

But enl file seems not be able to parsed directly, I load enl file in ENDNOTE app and exported as xml file

In [2]:
xml_path = "dictybase_files/dicty21/dicty21.xml"

tree = ET.parse(xml_path)
root = tree.getroot()

records = root.findall(".//record")
print("Total papers:", len(records))

Total papers: 10582


In [3]:
rec = root.find(".//record")  # first record

def dump(elem, depth=0, max_depth=6):
    if depth > max_depth:
        return
    txt = (elem.text or "").strip()
    if txt:
        txt = txt[:120]
    print("  "*depth + f"<{elem.tag}> {txt}")
    for ch in list(elem):
        dump(ch, depth+1, max_depth=max_depth)

dump(rec)



<record> 
  <database> dicty21.enl
  <source-app> EndNote
  <rec-number> 7482
  <foreign-keys> 
    <key> 7482
  <ref-type> 17
  <contributors> 
    <authors> 
      <author> 
        <style> Raper, K B
  <titles> 
    <title> 
      <style> Kenneth B. Raper papers 1925-1986
  <keywords> 
    <keyword> 
      <style> archives
  <dates> 
  <label> 
    <style> d7482
  <notes> 
    <style> 106 boxes (53.7 lin. ft); in the New York Botanical Garden library.
  <urls> 


In [4]:
pmid_re = re.compile(r"\bPMID[: ]*(\d{4,10})\b")

def style_texts(node):
    if node is None:
        return []
    styles = [s.text.strip() for s in node.findall(".//style") if s.text and s.text.strip()]
    if styles:
        return styles
    if node.text and node.text.strip():
        return [node.text.strip()]
    return []

def first_text_by_paths(rec, paths):
    """Try multiple XPath-like queries; return the first non-empty style/text."""
    for p in paths:
        node = rec.find(p)
        vals = style_texts(node)
        if vals:
            return vals[0]
    return ""

# candidate places where EndNote XML often stores abstracts
ABSTRACT_PATHS = [
    ".//abstract",                 # sometimes direct
    ".//abstracts/abstract",        # common
    ".//research-notes",            # sometimes used for abstract-ish text
    ".//notes",                     # fallback (may contain abstract or extra info)
]

rows = []

for event, rec in ET.iterparse(xml_path, events=("end",)):
    if rec.tag != "record":
        continue

    title = first_text_by_paths(rec, [".//titles/title"])
    year  = first_text_by_paths(rec, [".//dates/year"])
    rec_number = first_text_by_paths(rec, [".//rec-number", ".//key"])

    # authors (keep your current approach)
    author_nodes = rec.findall(".//contributors//author")
    authors = []
    for a in author_nodes:
        parts = style_texts(a)
        if parts:
            authors.append(parts[0])
    first_author = authors[0] if authors else ""

    # PMID (your logic)
    pmid = ""
    acc_txt = style_texts(rec.find(".//accession-num"))
    if acc_txt and acc_txt[0].isdigit():
        pmid = acc_txt[0]
    else:
        rec_str = ET.tostring(rec, encoding="unicode", method="xml")
        m = pmid_re.search(rec_str)
        if m:
            pmid = m.group(1)

    abstract = first_text_by_paths(rec, ABSTRACT_PATHS)

    rows.append({
        "rec_number": rec_number,
        "first_author": first_author,
        "year": year,
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
    })

    rec.clear()

df = pd.DataFrame(rows)
print("Total papers:", len(df))
print("With abstract:", df["abstract"].ne("").sum())

Total papers: 10582
With abstract: 2434


In [5]:
df

,rec_number,first_author,year,pmid,title,abstract
0,7482,"Raper, K B",,,Kenneth B. Raper papers 1925-1986,106 boxes (53.7 lin. ft); in the New York Bota...
1,8412,"Stephenson, S L",,,Methods for collecting eumycetozoan substrates...,
2,159,"Brefeld, O",1869,,Dictyostelium mucoroides. Ein neuer Organismus...,
3,1200,"Cienkowsky, L",1873,,Guttulina rosea.,\ article in Russian
4,1565,"van Tieghem, M P h",1880,,Sur quelques Myxomycetes a plasmode agrege.,
...,...,...,...,...,...,...
10577,10391,"Smith, J.",2016,26931797,Fine-scale spatial ecology drives kin selectio...,Cooperation among microbes is important for tr...
10578,10396,"Strassmann, J. E.",2016,26909677,Kin Discrimination in Dictyostelium Social Amo...,Evolved cooperation is stable only when the be...
10579,10460,"Tikhonenko, I.",2016,26298292,Organization of microtubule assemblies in Dict...,It has long been known that the interphase mic...
10580,10407,"Umeki, N.",2016,26842224,Cofilin-induced cooperative conformational cha...,To investigate cooperative conformational chan...


In [7]:
df.to_csv("dictybase_files/dicty21/dicty21.xml.txt", sep="\t", index=False)

In [8]:
# Year summary
df["year_num"] = pd.to_numeric(df["year"], errors="coerce")
year_counts = (df.dropna(subset=["year_num"])
                 .assign(year_num=lambda x: x["year_num"].astype(int))
                 .value_counts("year_num")
                 .sort_index())

print(year_counts.tail(20))  # latest 20 years in the library

# PMID coverage
missing_pmid = df["pmid"].eq("").sum()
print(f"Missing PMID: {missing_pmid} / {len(df)} ({missing_pmid/len(df):.1%})")

year_num
1997    290
1998    268
1999    241
2000    247
2001    239
2002    274
2003    205
2004    230
2005    196
2006    272
2007    221
2008    260
2009    201
2010    180
2011    212
2012    158
2013    187
2014    173
2015    167
2016     35
Name: count, dtype: int64
Missing PMID: 4441 / 10582 (42.0%)


In [9]:
df.loc[df["pmid"].ne(""), "pmid"].nunique()

6101

## xls file (excel with gene name and literatures)

In [11]:
df_map = pd.read_csv("dictybase_files/DDBID_PMID.csv",sep=";")
pmids_ddbid = set(df_map["pubmed"].dropna().astype(int).astype(str))
print("Unique PMIDs in DDBID_PMID:", len(pmids_ddbid))

Unique PMIDs in DDBID_PMID: 3840


In [12]:
df_map

,pubmed,gene_name,dictyBase id
0,11282973,DDB_G0267178_RTE,DDB0216437
1,16093681,DDB_G0267178_RTE,DDB0216437
2,26104693,DDB_G0267180_RTE,DDB0216438
3,11282973,DDB_G0267180_RTE,DDB0216438
4,16093681,DDB_G0267180_RTE,DDB0216438
...,...,...,...
22443,18366713,atp8,DDB0350612
22444,19861424,atp4,DDB0350620
22445,18366713,atp4,DDB0350620
22446,10821186,atp4,DDB0350620


### overlap between xml and csv(xls)

In [15]:
xml_pmids = (
    df.loc[df["pmid"].ne(""), "pmid"]
      .astype(str)
      .str.strip()
      .str.extract(r"(\d{4,10})", expand=False)   # keep just the digits
      .dropna()
)
xml_set = set(xml_pmids)

In [18]:
csv_pmids = (
    df_map["pubmed"]
      .dropna()
      .astype(str)
      .str.strip()
      .str.extract(r"(\d{4,10})", expand=False)
      .dropna()
)
csv_set = set(csv_pmids)

In [21]:
# --- 3) Overlap + differences ---
overlap = xml_set & csv_set
only_xml = xml_set - csv_set
only_csv = csv_set - xml_set

print("Overlap (unique PMIDs):", len(overlap))
print("Only in XML(df):", len(only_xml))
print("Only in CSV:", len(only_csv))

# optional: overlap percentages
print("Overlap as % of XML:", len(overlap)/len(xml_set) if xml_set else 0)
print("Overlap as % of CSV:", len(overlap)/len(csv_set) if csv_set else 0)

Overlap (unique PMIDs): 2689
Only in XML(df): 3412
Only in CSV: 1150
Overlap as % of XML: 0.4407474184559908
Overlap as % of CSV: 0.7004428236519927


## webpage

the above two files do not have all publications on dictybase, also it is diffcult to get internal publication ids there, so just ignore.

it is possible to browser all pages of http://dictybase.org/publication/xxxx but there server is down often

e.g http://dictybase.org/publication/19729

In the end i used gene_publication_mapping.ipynb and scripts/public/dicty_publication.py, works good

# check pmids in fetched from EPMC and dictybase

In [54]:
from pathlib import Path
import json
import polars as pl

## jsons of EPMC

In [55]:
DATA_DIR = Path("scripts/public/article_fetching/output/all_cleaned")

In [56]:
rows = []
for fp in DATA_DIR.rglob("*.json"):
    try:
        with fp.open("r", encoding="utf-8") as f:
            d = json.load(f)

        abstract = d.get("abstract", None)
        text = d.get("text", None)

        # has_abstract: not null and not empty
        has_abstract = abstract is not None and str(abstract).strip() != ""

        # has_text: not null and contains some content
        if text is None:
            has_text = False
            n_text_sections = 0
            n_text_paras = 0
        elif isinstance(text, dict):
            n_text_sections = len(text)
            n_text_paras = sum(len(v) for v in text.values() if isinstance(v, list))
            has_text = (n_text_sections > 0) and (n_text_paras > 0)
        else:
            # in case text is stored as a string in some files
            has_text = str(text).strip() != ""
            n_text_sections = None
            n_text_paras = None

        rows.append({
            "pmid": d.get("pmid"),
            "pmcid": d.get("pmcid"),
            "doi": d.get("doi"),
            "year": d.get("year"),
            "title": d.get("title"),
            "journal": d.get("journal"),
            "authors": d.get("authors"),
            "has_abstract": has_abstract,
            "has_text": has_text,
            "n_text_sections": n_text_sections,
            "n_text_paras": n_text_paras,
            "file": str(fp),
        })

    except Exception as e:
        rows.append({
            "pmid": None,
            "pmcid": None,
            "doi": None,
            "year": None,
            "title": None,
            "journal": None,
            "authors": None,
            "has_abstract": False,
            "has_text": False,
            "n_text_sections": None,
            "n_text_paras": None,
            "file": str(fp),
            "error": str(e),
        })

df = pl.DataFrame(rows)

In [57]:
# your "pmid table"
pmid_table = (
    df.select(["pmid", "pmcid", "doi", "year", "title", "journal", "authors", "file"])
      .filter(pl.col("pmid").is_not_null())
)

In [58]:
pmid_table

pmid,pmcid,doi,year,title,journal,authors,file
str,str,str,str,str,str,str,str
"""2654141""","""PMC2115546""","""10.1083/jcb.108.5.1751""","""1989""","""Centrin-mediated microtubule s…","""The Journal of cell biology""","""Sanders MA, Salisbury JL.""","""article_fetching/output/all_cl…"
"""39528565""","""PMC11555045""","""10.1038/s41467-024-54272-4""","""2024""","""Nuclear localization sequence …","""Nature communications""","""Lim YJ, Yoon YJ, Lee H, Choi G…","""article_fetching/output/all_cl…"
"""6319129""",null,"""10.1111/j.1432-1033.1984.tb078…","""1984""","""Investigations on stimulation …","""European journal of biochemist…","""Scholübbers HG, van Knippenber…","""article_fetching/output/all_cl…"
"""28740830""","""PMC5502327""","""10.3389/fonc.2017.00139""","""2017""","""Structure, Activity Regulation…","""Frontiers in oncology""","""Mammucari C, Gherardi G, Rizzu…","""article_fetching/output/all_cl…"
"""18814278""",null,"""10.1002/cm.20314""","""2008""","""Correlated waves of actin fila…","""Cell motility and the cytoskel…","""Asano Y, Nagasaki A, Uyeda TQ.""","""article_fetching/output/all_cl…"
…,…,…,…,…,…,…,…
"""36543032""","""PMC9889102""","""10.1016/j.ejmech.2022.115008""","""2023""","""Isoform selectivities of novel…","""European journal of medicinal …","""Smith JD, Brawley J, Bordenave…","""article_fetching/output/all_cl…"
"""20023070""","""PMC2823002""","""10.1128/ec.00220-09""","""2010""","""Distinct subcellular localizat…","""Eukaryotic cell""","""Schilde C, Schönemann B, Sehri…","""article_fetching/output/all_cl…"
"""10706824""",null,"""10.1006/scdb.1999.0343""","""1999""","""Control of spatial patterning …","""Seminars in cell & development…","""Mohanty S, Firtel RA.""","""article_fetching/output/all_cl…"


## check how this overlaps all publications on dictybase that has pmids

In [59]:
dicty_pmid_csv = "output/publication_id_pmid.csv"
dicty_pmid = pl.read_csv(dicty_pmid_csv)

In [60]:
dicty_pmid

publication_id,pmid
i64,i64
12,21243421
13,21239624
15,21235525
17,20950684
20,21150268
…,…
19689,32769116
19708,21551065
19728,32821814


In [61]:
def normalize_pmid_expr(col: str) -> pl.Expr:
    return (
        pl.col(col)
        .cast(pl.Utf8, strict=False)
        .str.strip_chars()
        .str.replace(r"\.0$", "", literal=False)   # Excel float artifact
        .str.extract(r"(\d+)", 1)                 # keep digits only (first group)
    )

pmids_epmc = (
    pmid_table
    .select(pmid=normalize_pmid_expr("pmid"))
    .drop_nulls()
    .unique()
)

pmids_dictybase = (
    dicty_pmid
    .select(pmid=normalize_pmid_expr("pmid"))
    .drop_nulls()
    .unique()
)

In [62]:
overlap = pmids_epmc.join(pmids_dictybase, on="pmid", how="inner")
only_epmc  = pmids_epmc.join(pmids_dictybase, on="pmid", how="anti")
only_dictybase  = pmids_dictybase.join(pmids_epmc, on="pmid", how="anti")

In [63]:
summary = pl.DataFrame({
    "set": ["pmid_table", "dicty_pmid", "overlap", "only_epmc", "only_dictybase"],
    "unique_count": [pmids_epmc.height, pmids_dictybase.height, overlap.height, only_epmc.height, only_dictybase.height],
})

summary

set,unique_count
str,i64
"""pmid_table""",21986
"""dicty_pmid""",4324
"""overlap""",4289
"""only_epmc""",17697
"""only_dictybase""",35


it seems that many publication on dictybase were not fetched on epmc, let look at them

In [64]:
only_dictybase.head()

pmid
str
"""24115140"""
"""31924893"""
"""15196561"""
"""17289020"""
"""24690554"""


In the case, that the query is "Dictyostelium", we almost cover all the literature on dictybase, only 35 was missing. 
This looks good!

Earlier, when I query "OPEN_ACCESS:y AND \"Dictyostelium discoideum\", it covers only a bit more than 10%.

We shall still look at all the literatues and see if they have abstract.


## check in all literatures fetched, how many have abstract and text

In [65]:
# quick summary
summary = df.select([
    pl.len().alias("n_files"),
    pl.col("pmid").is_not_null().sum().alias("n_with_pmid"),
    pl.col("has_abstract").sum().alias("n_with_abstract"),
    pl.col("has_text").sum().alias("n_with_text"),
    (pl.col("has_abstract") & pl.col("has_text")).sum().alias("n_with_both"),
    (~pl.col("has_abstract") & ~pl.col("has_text")).sum().alias("n_with_neither"),
])

summary

n_files,n_with_pmid,n_with_abstract,n_with_text,n_with_both,n_with_neither
u32,u32,u32,u32,u32,u32
22963,21986,20951,7892,7569,1689


 ## check in all literatures fetched that also on dictybase, how many have abstract and text

In [66]:
# 1) restrict df to overlap PMIDs
df_overlap = df.join(overlap, on="pmid", how="inner")

# 2) counts + fractions within overlap
overlap_summary = df_overlap.select([
    pl.len().alias("n_overlap_pmids"),
    pl.col("has_abstract").sum().alias("n_with_abstract"),
    pl.col("has_text").sum().alias("n_with_text"),
    (pl.col("has_abstract") & pl.col("has_text")).sum().alias("n_with_both"),
    (~pl.col("has_abstract") & ~pl.col("has_text")).sum().alias("n_with_neither"),
    (pl.col("has_abstract").mean()).alias("frac_with_abstract"),
    (pl.col("has_text").mean()).alias("frac_with_text"),
    ((pl.col("has_abstract") & pl.col("has_text")).mean()).alias("frac_with_both"),
])

overlap_summary

n_overlap_pmids,n_with_abstract,n_with_text,n_with_both,n_with_neither,frac_with_abstract,frac_with_text,frac_with_both
u32,u32,u32,u32,u32,f64,f64,f64
4289,4164,675,673,123,0.970856,0.157379,0.156913


at least most of them have abstract!

## let us save the table so we can use in datasets.ipynb

In [84]:
pmid_uniqueness = pmid_table.select([
    pl.col("pmid").is_not_null().sum().alias("n_nonnull_pmid"),
    pl.col("pmid").filter(pl.col("pmid").is_not_null()).n_unique().alias("n_unique_nonnull_pmid"),
]).with_columns(
    (pl.col("n_nonnull_pmid") - pl.col("n_unique_nonnull_pmid")).alias("n_duplicate_nonnull_pmid")
)

pmid_uniqueness

n_nonnull_pmid,n_unique_nonnull_pmid,n_duplicate_nonnull_pmid
u32,u32,u32
21986,21986,0


ok, no missing and duplicated pmids.

In [87]:
rows = []

for fp in DATA_DIR.rglob("*.json"):
    try:
        with fp.open("r", encoding="utf-8") as f:
            d = json.load(f)

        abstract = d.get("abstract", None)

        # normalize abstract to a string
        if abstract is None:
            continue
        if isinstance(abstract, list):
            abstract_str = "\n".join(str(x).strip() for x in abstract if str(x).strip() != "")
        else:
            abstract_str = str(abstract).strip()

        if abstract_str == "":
            continue  # only keep those that truly have abstracts

        text = d.get("text", None)

        # normalize text into two representations:
        # 1) nested text dict 그대로 (good for Parquet)
        text_nested = text if isinstance(text, dict) else None

        # 2) flattened plain text (useful for search / TSV)
        if text is None:
            text_plain = None
        elif isinstance(text, dict):
            parts = []
            for section, paras in text.items():
                if isinstance(paras, list):
                    # keep section header optionally
                    # parts.append(f"## {section}")
                    parts.extend([str(p).strip() for p in paras if str(p).strip() != ""])
            text_plain = "\n".join(parts).strip() if parts else None
        else:
            text_plain = str(text).strip() or None

        rows.append({
            "pmid": d.get("pmid"),
            "pmcid": d.get("pmcid"),
            "doi": d.get("doi"),
            "year": d.get("year"),
            "title": d.get("title"),
            "journal": d.get("journal"),
            "authors": d.get("authors"),
            "abstract": abstract_str,      # REAL abstract text
            "text_plain": text_plain,      # flattened full text (optional but handy)
            "text": text_nested,           # nested full text dict (optional; parquet only)
            "file": str(fp),
        })

    except Exception as e:
        # skip broken files (or keep them if you want)
        continue

df_abs = pl.DataFrame(rows)
df_abs = (
    df_abs.filter(pl.col("pmid").is_not_null())
)
df_abs

shape: (20_447, 11)
┌──────────┬────────────┬────────────┬──────┬───┬────────────┬────────────┬────────────┬───────────┐
│ pmid     ┆ pmcid      ┆ doi        ┆ year ┆ … ┆ abstract   ┆ text_plain ┆ text       ┆ file      │
│ ---      ┆ ---        ┆ ---        ┆ ---  ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│ str      ┆ str        ┆ str        ┆ str  ┆   ┆ str        ┆ str        ┆ struct[493 ┆ str       │
│          ┆            ┆            ┆      ┆   ┆            ┆            ┆ ]          ┆           │
╞══════════╪════════════╪════════════╪══════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 2654141  ┆ PMC2115546 ┆ 10.1083/jc ┆ 1989 ┆ … ┆ Chlamydomo ┆ Centrin-me ┆ {["Centrin ┆ article_f │
│          ┆            ┆ b.108.5.17 ┆      ┆   ┆ nas cells  ┆ diated mic ┆ -mediated  ┆ etching/o │
│          ┆            ┆ 51         ┆      ┆   ┆ excise     ┆ rotubule   ┆ microtubul ┆ utput/all │
│          ┆            ┆            ┆      ┆   ┆ the…       ┆ s…         ┆ …          ┆ _cl…      │
│ 39528565 ┆ PMC1155504 ┆ 10.1038/s4 ┆ 2024 ┆ … ┆ Plant      ┆ Nuclear    ┆ {["Nuclear ┆ article_f │
│          ┆ 5          ┆ 1467-024-5 ┆      ┆   ┆ pathogens  ┆ localizati ┆ localizati ┆ etching/o │
│          ┆            ┆ 4272-4     ┆      ┆   ┆ secrete    ┆ on         ┆ on sequen… ┆ utput/all │
│          ┆            ┆            ┆      ┆   ┆ nuclea…    ┆ sequence … ┆            ┆ _cl…      │
│ 6319129  ┆ null       ┆ 10.1111/j. ┆ 1984 ┆ … ┆ The        ┆ null       ┆ null       ┆ article_f │
│          ┆            ┆ 1432-1033. ┆      ┆   ┆ ability of ┆            ┆            ┆ etching/o │
│          ┆            ┆ 1984.tb078 ┆      ┆   ┆ 24 systema ┆            ┆            ┆ utput/all │
│          ┆            ┆ …          ┆      ┆   ┆ tical…     ┆            ┆            ┆ _cl…      │
│ 28740830 ┆ PMC5502327 ┆ 10.3389/fo ┆ 2017 ┆ … ┆ Mitochondr ┆ Structure, ┆ {["Structu ┆ article_f │
│          ┆            ┆ nc.2017.00 ┆      ┆   ┆ ial Ca<sup ┆ Activity   ┆ re,        ┆ etching/o │
│          ┆            ┆ 139        ┆      ┆   ┆ >2+</sup>  ┆ Regulation ┆ Activity   ┆ utput/all │
│          ┆            ┆            ┆      ┆   ┆ …          ┆ …          ┆ Regulat…   ┆ _cl…      │
│ 18814278 ┆ null       ┆ 10.1002/cm ┆ 2008 ┆ … ┆ Chemotaxis ┆ null       ┆ null       ┆ article_f │
│          ┆            ┆ .20314     ┆      ┆   ┆ -deficient ┆            ┆            ┆ etching/o │
│          ┆            ┆            ┆      ┆   ┆ amiB-null… ┆            ┆            ┆ utput/all │
│          ┆            ┆            ┆      ┆   ┆            ┆            ┆            ┆ _cl…      │
│ …        ┆ …          ┆ …          ┆ …    ┆ … ┆ …          ┆ …          ┆ …          ┆ …         │
│ 36543032 ┆ PMC9889102 ┆ 10.1016/j. ┆ 2023 ┆ … ┆ Muscle     ┆ null       ┆ null       ┆ article_f │
│          ┆            ┆ ejmech.202 ┆      ┆   ┆ myosin     ┆            ┆            ┆ etching/o │
│          ┆            ┆ 2.115008   ┆      ┆   ┆ inhibition ┆            ┆            ┆ utput/all │
│          ┆            ┆            ┆      ┆   ┆ could…     ┆            ┆            ┆ _cl…      │
│ 20023070 ┆ PMC2823002 ┆ 10.1128/ec ┆ 2010 ┆ … ┆ We have    ┆ null       ┆ null       ┆ article_f │
│          ┆            ┆ .00220-09  ┆      ┆   ┆ identified ┆            ┆            ┆ etching/o │
│          ┆            ┆            ┆      ┆   ┆ new        ┆            ┆            ┆ utput/all │
│          ┆            ┆            ┆      ┆   ┆ synapto…   ┆            ┆            ┆ _cl…      │
│ 10706824 ┆ null       ┆ 10.1006/sc ┆ 1999 ┆ … ┆ The        ┆ null       ┆ null       ┆ article_f │
│          ┆            ┆ db.1999.03 ┆      ┆   ┆ spatial    ┆            ┆            ┆ etching/o │
│          ┆            ┆ 43         ┆      ┆   ┆ patterning ┆            ┆            ┆ utput/all │
│          ┆            ┆            ┆      ┆   ┆ of pres…   ┆            ┆            ┆ _cl…      │
│ 39396247 ┆ PMC1148569 ┆ 10.1080/19 ┆ 20

In [81]:
# (
#     df_abs
#     .drop(["text", "text_plain"])
#     .head(1000)
#     .write_csv("output/pmid_table_with_abstract.tsv", separator="\t")
# )

let us clean up HTML/JATS markup

In [88]:
def unescape(s: str | None) -> str | None:
    return None if s is None else html.unescape(s)

df_abs_clean = df_abs.with_columns(
    pl.col("abstract")
    # decode entities like &amp; &lt; etc
    .map_elements(unescape, return_dtype=pl.Utf8)
    # add newlines after headings / paragraphs / breaks
    .str.replace_all(r"</h[1-6]>", "\n")
    .str.replace_all(r"</p>", "\n")
    .str.replace_all(r"<br\s*/?>", "\n")
    # drop heading open tags (<h4 ...>) and all remaining tags (<i>, <sup>, <sub>, ...)
    .str.replace_all(r"<h[1-6][^>]*>", "")
    .str.replace_all(r"</?[^>]+>", "")
    # whitespace normalization
    .str.replace_all(r"\r\n", "\n")
    .str.replace_all(r"[ \t]+", " ")
    .str.replace_all(r"\n{3,}", "\n\n")
    .str.strip_chars()
    .alias("abstract_clean")
)

In [89]:
with_tags = df_abs.filter(pl.col("abstract").str.contains(r"<[^>]+>")).head(5)

with pl.Config(fmt_str_lengths=2000):
    display(
        df_abs_clean.join(with_tags.select("pmid"), on="pmid", how="inner")
                   .select(["pmid", "abstract", "abstract_clean"])
    )

pmid,abstract,abstract_clean
str,str,str
"""28740830""","""Mitochondrial Ca<sup>2+</sup> uptake plays a pivotal role both in cell energy balance and in cell fate determination. Studies on the role of mitochondrial Ca<sup>2+</sup> signaling in pathophysiology have been favored by the identification of the genes encoding the mitochondrial calcium uniporter (MCU) and its regulatory subunits. Thus, research carried on in the last years on one hand has determined the structure of the MCU complex and its regulation, on the other has uncovered the consequences of dysregulated mitochondrial Ca<sup>2+</sup> signaling in cell and tissue homeostasis. Whether mitochondrial Ca<sup>2+</sup> uptake can be exploited as a weapon to counteract cancer progression is debated. In this review, we summarize recent research on the molecular structure of the MCU, the regulatory mechanisms that control its activity and its relevance in pathophysiology, focusing in particular on its role in cancer progression.""","""Mitochondrial Ca2+ uptake plays a pivotal role both in cell energy balance and in cell fate determination. Studies on the role of mitochondrial Ca2+ signaling in pathophysiology have been favored by the identification of the genes encoding the mitochondrial calcium uniporter (MCU) and its regulatory subunits. Thus, research carried on in the last years on one hand has determined the structure of the MCU complex and its regulation, on the other has uncovered the consequences of dysregulated mitochondrial Ca2+ signaling in cell and tissue homeostasis. Whether mitochondrial Ca2+ uptake can be exploited as a weapon to counteract cancer progression is debated. In this review, we summarize recent research on the molecular structure of the MCU, the regulatory mechanisms that control its activity and its relevance in pathophysiology, focusing in particular on its role in cancer progression."""
"""35602934""","""The social ameba <i>Dictyostelium discoideum</i> has emerged as a powerful model to study mitochondrial genetics and bioenergetics. However, a comprehensive inventory of mitochondrial proteins that is critical to understanding mitochondrial processes has yet to be curated. Here, we utilized high-throughput multiplexed protein quantitation and homology analyses to generate a high-confidence mitochondrial protein compendium consisting of 936 proteins. Our proteomic approach, which utilizes mass spectrometry in combination with mathematical modeling, was validated through mitochondrial targeting sequence prediction and live-cell imaging. Our final compendium consists of 936 proteins. Nearly, a third of <i>D. discoideum</i> mitochondrial proteins do not have homologs in humans, budding yeasts, or an ancestral alphaproteobacteria. Additionally, we leverage our compendium to highlight the complexity of metabolic reprogramming during starvation-induced development. Our compendium lays a foundation to investigate mitochondrial processes that are unique in ameba and to understand the functions of conserved mitochondrial proteins in <i>D. discoideum</i>.""","""The social ameba Dictyostelium discoideum has emerged as a powerful model to study mitochondrial genetics and bioenergetics. However, a comprehensive inventory of mitochondrial proteins that is critical to understanding mitochondrial processes has yet to be curated. Here, we utilized high-throughput multiplexed protein quantitation and homology analyses to generate a high-confidence mitochondrial protein compendium consisting of 936 proteins. Our proteomic approach, which utilizes mass spectrometry in combination with mathematical modeling, was validated through mitochondrial targeting sequence prediction and live-cell imaging. Our final compendium consists of 936 proteins. Nearly, a third of D. discoideum mitochondrial proteins do not have homologs in humans, budding yeasts, or an ancestral alphaproteobacteria. Additionally, we leverage our compendium to highlight the complexity of metabolic rep

In [90]:
df_abs_clean_small = df_abs_clean.select([
    "pmid",
    "pmcid",
    "doi",
    "year",
    "title",
    "journal",
    "authors",
    "abstract_clean",
    "file",
])

df_abs_clean_small.write_parquet("output/cleaned/articles_all_cleaned_abstract.parquet")